# CircRNA Flanking Intron Motif Discovery
**Course**: Introduction to Bioinformatics (MBIO0402), 2026  
**Kyungpook National University**  

This notebook implements the complete pipeline for the circRNA flanking intron motif discovery project.  

### Pipeline Stages:
1. **Step 1: log2 Fold Change Calculation & Labeling** (UP / DOWN / SAME)
2. **Step 2: Flanking Intron Coordinate Extraction** (using RefSeq GTF)
3. **Step 3: BED File Generation** (up_I1, up_I2, down_I1, down_I2)
4. **Step 4: FASTA Sequence Extraction** (from GRCm39 reference genome)
5. **Step 5: Motif Discovery (MEME Suite)** (Instructions for web execution)
6. **Step 6: Motif Comparison (TOMTOM)** (Instructions for web execution)

### Step 1: log2 Fold Change Calculation & Labeling
We compute the log2 Fold Change of circRNA expression at week 8 compared to week 1:
$$\log_2 FC = \log_2 \left( \frac{\text{circ\_tac\_8wk}}{\text{circ\_tac\_1wk}} \right)$$

Using a threshold of 0.5, we classify each circRNA as:
- **UP**: $\log_2 FC > 0.5$ (Up-regulated at 8 weeks)
- **DOWN**: $\log_2 FC < -0.5$ (Down-regulated at 8 weeks)
- **SAME**: $-0.5 \le \log_2 FC \le 0.5$ (No significant change)

In [ ]:
# 1. Load circRNAs and calculate log2FC
print("Step 1: Calculating log2 Fold Changes...")
circrnas = []
target_genes = set()

with open(csv_path, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for idx, row in enumerate(reader):
        c1 = float(row["circ_tac_1wk"])
        c8 = float(row["circ_tac_8wk"])
        log2fc = math.log2(c8 / c1)
        
        # Parse circRNA_ID (format: chr8:85498413|85498944)
        m = re.match(r"([^:]+):(\d+)\|(\d+)", row["circRNA_ID"])
        if not m:
            raise ValueError(f"Invalid circRNA_ID format: {row['circRNA_ID']}")
        chrom = m.group(1)
        start = int(m.group(2))
        end = int(m.group(3))
        
        circ = {
            "circRNA_ID": row["circRNA_ID"],
            "host_gene": row["host_gene"],
            "strand": row["strand"],
            "circ_tac_1wk": row["circ_tac_1wk"],
            "circ_tac_8wk": row["circ_tac_8wk"],
            "log2FC": log2fc,
            "chrom": chrom,
            "circ_start": start,
            "circ_end": end,
            "file_idx": idx
        }
        
        # Labeling (threshold = 0.5)
        if log2fc > 0.5:
            circ["label"] = "UP"
        elif log2fc < -0.5:
            circ["label"] = "DOWN"
        else:
            circ["label"] = "SAME"
            
        circrnas.append(circ)
        target_genes.add(row["host_gene"])

# Save labeled circRNAs to top300_1.csv
with open(csv_out_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["circRNA_ID", "host_gene", "strand", "circ_tac_1wk", "circ_tac_8wk", "log2FC", "label"])
    for circ in circrnas:
        writer.writerow([
            circ["circRNA_ID"],
            circ["host_gene"],
            circ["strand"],
            circ["circ_tac_1wk"],
            circ["circ_tac_8wk"],
            f"{circ['log2FC']:.6f}",
            circ["label"]
        ])

print(f"Labeled circRNAs written to {csv_out_path}")
print(f"Total target genes: {len(target_genes)}")


### Step 2 & 3: Flanking Intron Coordinate Extraction & BED File Generation
We parse `mm39.ncbiRefSeq.gtf` to obtain the exon-intron structures of the host genes and extract the precise flanking intron boundaries.

- **I1 (upstream intron)** and **I2 (downstream intron)** boundaries are assigned according to the transcript strand direction:
  - **Forward strand (+)**: I1 is the left flanking intron (upstream of start); I2 is the right flanking intron (downstream of end).
  - **Reverse strand (-)**: I1 is the right flanking intron (upstream of end); I2 is the left flanking intron (downstream of start).

- **Boundary Rules**:
  - If the flanking intron is $\le 500$ bp, extract the full intron.
  - If the intron is $> 500$ bp, extract up to 500 bp starting from the BSJ-adjacent end.

In [ ]:
# 2. Parse GTF for exons of target genes
print("\nStep 2: Parsing GTF for exon coordinates (optimized)...")
gene_exons = {gene: [] for gene in target_genes}

with open(gtf_path, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split("\t")
        if len(parts) < 9 or parts[2] != "exon":
            continue
            
        attrs = parts[8]
        # Quick check if any target gene name is in attributes to avoid regex when possible
        has_target = False
        for g in target_genes:
            if g in attrs:
                has_target = True
                break
        if not has_target:
            continue
            
        m = re.search(r'gene_name "([^"]+)"', attrs)
        if not m:
            m = re.search(r'gene_id "([^"]+)"', attrs)
        if m:
            g_name = m.group(1)
            if g_name in target_genes:
                chrom = parts[0]
                start = int(parts[3])
                end = int(parts[4])
                strand = parts[6]
                gene_exons[g_name].append((chrom, start, end, strand))

# Deduplicate and sort exons
for gene in gene_exons:
    gene_exons[gene] = sorted(list(set(gene_exons[gene])), key=lambda x: x[1])

print("GTF parsing completed.")

# 3. Extract flanking intron coordinates and generate BED records
print("\nStep 3: Extracting flanking intron coordinates...")
bed_records = {
    "up_I1": [],
    "up_I2": [],
    "down_I1": [],
    "down_I2": []
}

for circ in circrnas:
    label = circ["label"]
    if label == "SAME":
        continue
    
    gene = circ["host_gene"]
    chrom = circ["chrom"]
    circ_start = circ["circ_start"]
    circ_end = circ["circ_end"]
    strand = circ["strand"]
    file_idx = circ["file_idx"]
    
    exons = gene_exons.get(gene, [])
    # Filter for exons matching chromosome and strand
    exons = [e for e in exons if e[0] == chrom and e[3] == strand]
    
    # Left intron flanking circ_start (preceding exon end to circ_start - 1)
    preceding_ends = [e[2] for e in exons if e[2] <= circ_start]
    E = max(preceding_ends) if preceding_ends else None
    
    left_len = circ_start - E if E is not None else 999999
    if left_len > 500:
        left_start = circ_start - 1 - 500
        left_end = circ_start - 1
    else:
        left_start = E
        left_end = circ_start - 1
        
    # Right intron flanking circ_end (circ_end to succeeding exon start - 1)
    succeeding_starts = [e[1] for e in exons if e[1] >= circ_end]
    S = min(succeeding_starts) if succeeding_starts else None
    
    right_len = S - circ_end if S is not None else 999999
    if right_len > 500:
        right_start = circ_end
        right_end = circ_end + 500
    else:
        right_start = circ_end
        right_end = S - 1
        
    # Assign I1 and I2 based on strand
    if strand == "+":
        i1_start, i1_end = left_start, left_end
        i2_start, i2_end = right_start, right_end
    else:
        i1_start, i1_end = right_start, right_end
        i2_start, i2_end = left_start, left_end
        
    # Region name format: Gene_FileIndex_Intron
    i1_name = f"{gene}_{file_idx}_I1"
    i2_name = f"{gene}_{file_idx}_I2"
    
    # BED record format: chrom, start_0, end_0, name, score, strand
    key_prefix = "up" if label == "UP" else "down"
    
    bed_records[f"{key_prefix}_I1"].append([chrom, i1_start, i1_end, i1_name, 0, strand])
    bed_records[f"{key_prefix}_I2"].append([chrom, i2_start, i2_end, i2_name, 0, strand])

# Write BED files
for key, records in bed_records.items():
    out_file = bed_files[key]
    with open(out_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter="\t")
        for rec in records:
            writer.writerow(rec)
    print(f"Generated BED file: {out_file} ({len(records)} records)")


### Step 4: FASTA Sequence Extraction
We extract the actual nucleotide sequences from the reference genome (`mm39.fa`).

#### Equivalent command line with `bedtools getfasta`:
```bash
bedtools getfasta -fi mm39.fa -bed up_I1.bed -fo up_I1.fa -s -name
bedtools getfasta -fi mm39.fa -bed up_I2.bed -fo up_I2.fa -s -name
bedtools getfasta -fi mm39.fa -bed down_I1.bed -fo down_I1.fa -s -name
bedtools getfasta -fi mm39.fa -bed down_I2.bed -fo down_I2.fa -s -name
```

Below is the optimized, chromosome-buffered Python implementation that executes inside the Jupyter environment without needing conda/external dependencies.

In [ ]:
# 4. Extract sequences from reference genome and write to FASTA files
print("\nStep 4: Extracting FASTA sequences from mm39.fa (optimized)...")

# Build map of needed regions
needed_regions = {}
for key, records in bed_records.items():
    for rec in records:
        chrom, start_0, end_0, name, _, strand = rec
        if chrom not in needed_regions:
            needed_regions[chrom] = []
        needed_regions[chrom].append((start_0, end_0, name, strand, key))

comp_dict = {"A":"T", "T":"A", "C":"G", "G":"C", "N":"N",
             "a":"t", "t":"a", "c":"g", "g":"c", "n":"n"}

def reverse_complement(s):
    return "".join(comp_dict.get(base, base) for base in reversed(s))

fasta_data = {key: [] for key in bed_records}

def process_chrom_buffer(chrom, lines):
    if not lines or chrom not in needed_regions:
        return
    print(f"Extracting regions for chromosome {chrom}...")
    full_seq = "".join(lines)
    for start_0, end_0, name, strand, key in needed_regions[chrom]:
        seq = full_seq[start_0:end_0]
        if len(seq) != (end_0 - start_0):
            print(f"Warning: Extracted length {len(seq)} for {name} doesn't match expected {end_0 - start_0}")
        if strand == "-":
            seq = reverse_complement(seq)
        fasta_data[key].append(f">{name} {chrom}:{start_0}|{end_0} len={len(seq)}\n{seq}\n")

current_chrom = None
chrom_lines = []

with open(fa_path, "r") as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
            if current_chrom and current_chrom in needed_regions:
                process_chrom_buffer(current_chrom, chrom_lines)
            
            new_chrom = line.split()[0][1:]
            if new_chrom in needed_regions:
                current_chrom = new_chrom
                chrom_lines = []
                print(f"Reading chromosome {new_chrom} lines...")
            else:
                current_chrom = None
                chrom_lines = []
        else:
            if current_chrom:
                chrom_lines.append(line)

if current_chrom and current_chrom in needed_regions:
    process_chrom_buffer(current_chrom, chrom_lines)

# Write FASTA files
for key, lines in fasta_data.items():
    out_file = fa_files[key]
    with open(out_file, "w", encoding="utf-8") as f:
        f.writelines(lines)
    print(f"Generated FASTA file: {out_file} ({len(lines)} sequences)")

print("\nPipeline completed successfully!")


### Step 5: MEME Motif Discovery
To search for de novo sequence motifs in the flanking introns, upload the 4 generated FASTA files to the MEME Suite website:

1. Open [https://meme-suite.org/meme/tools/meme](https://meme-suite.org/meme/tools/meme) in your browser.
2. Upload the respective FASTA files one by one (`up_I1.fa`, `up_I2.fa`, `down_I1.fa`, `down_I2.fa`).
3. Set the following configuration parameters:
   - **Sequence alphabet**: DNA, RNA or Protein (Auto-detect)
   - **Discovery mode**: Classic
   - **Select the site distribution**: *Any Number of Repetitions (anr)*
   - **Select the number of motifs**: *10*
   - **Motif width (Advanced options)**: Min = *10*, Max = *10*
4. Submit the job and download the raw text result as `up_I1_meme.txt`, `up_I2_meme.txt`, `down_I1_meme.txt`, or `down_I2_meme.txt` once complete.

### Step 6: Motif Comparison (TOMTOM)
To identify common regulatory elements and specific motifs distinguishing UP-regulated and DOWN-regulated flanking introns, we compare the discovered motif sets using TOMTOM:

1. Open [https://meme-suite.org/meme/tools/tomtom](https://meme-suite.org/meme/tools/tomtom) in your browser.
2. **I1 (Upstream Intron) Comparison**:
   - **Query motifs**: Upload `up_I1_meme.txt`
   - **Target motifs**: Upload `down_I1_meme.txt`
3. **I2 (Downstream Intron) Comparison**:
   - **Query motifs**: Upload `up_I2_meme.txt`
   - **Target motifs**: Upload `down_I2_meme.txt`
4. Set the **Significance threshold** to: *E-value < 0.01*.
5. Start the search. Overlapping motifs identify shared splicing factors or common elements, while non-overlapping motifs represent condition-specific regulatory features.